Sequential Chats and Customer Onboarding

In [1]:
from utils import get_hugging_face_api_key
HUGGING_FACE_API_KEY= get_hugging_face_api_key()
llm_config=  {
            "model": "meta-llama/Meta-Llama-3-8B-Instruct",
            "api_key": HUGGING_FACE_API_KEY,
            "base_url": "https://router.huggingface.co/v1",
            "api_type": "openai"
            }

In [2]:
from autogen import ConversableAgent

C:\Anaconda\envs\aiagent\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


Creating the needed agents

In [3]:
onboarding_personal_information_agent= ConversableAgent(
    name="onboarding_personal_information_agent",
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config= llm_config,
    code_execution_config= False,
    human_input_mode= "NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


Creating tasks


Now, you can craft a series of tasks to facilitate the onboarding process.

In [12]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location? Return the customer information as JSON object only: {'name': '', 'location': ''",
        "summary_method": "last_msg",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True,
        "silent": True,
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "last_msg",
        "max_turns": 1,
        "clear_history": False,
        "silent": True,
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1, 
        "summary_method": "last_msg",
    },
]

Start the onboarding process

Note: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [13]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)

C:\Anaconda\envs\aiagent\Lib\site-packages\autogen\agentchat\chat.py:47: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  faiz from thanjavur
Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Flower



********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
faiz from thanjavur
Flower

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Hi Faiz from Thanjavur!

I've got a lovely surprise for you! Since you mentioned "Flower", I'd like to share an interesting fact with you. Did you know that Thanjavur is also known for its beautiful Marigold flowers? In fact, Thanjavur is one of the largest producers of Marigold flowers in India!

Here's a fun fact: Marigold flowers are also known as "Genda" in Hindi, and they're often used in traditional Indian weddings as a symbol of prosperity and good luck!

Would you like to hear more fun facts or jokes? Or maybe you'd like to explore more 